# Part 3 · Acoustic reconstruction from CIS pulses

**Sensory Prostheses Engineering · BME 576**  
Gianluigi D'Antonio · Gabriele Colò · Yelizaveta Semikina

This temporary notebook runs independently of `cochlear_implant_assignment.ipynb`.
It calls the existing modules for parts 1 and 2, then develops the acoustic
reconstruction step by step. Place it in the repository root next to the original notebook.

The assignment asks us to attempt to recreate speech and music from four electrical
signals. Here, the decoder reads the **CIS pulse trains**, recovers their envelope
samples and uses them to modulate band-limited noise. Original envelopes are kept
only for verification. Original audio is used only as input to the encoder and as
a listening reference.

This is an acoustic illustration of information preserved by our simplified CIS
model. It is not an exact recovery of the waveform or a prediction of an implanted
listener's perception. Temporal fine structure and phase are not transmitted.

Run all cells in order with the project's Python environment. The first run of
`librosa.ex` downloads the example recordings if they are not already cached.

## 1. Imports and shared parameters

The analysis bands, envelope cutoff and compression parameter match part 1.
The pulse parameters match Liz's part 2. The audio clock and stimulation clock
are different and must remain distinct throughout decoding.

| Parameter | Value |
|---|---|
| Audio sampling rate | 22,050 Hz |
| Band edges | 200, 500, 1250, 3150, 8000 Hz |
| Envelope low-pass cutoff | 400 Hz |
| Compression alpha | 1000 |
| Stimulation sampling rate | 100,000 Hz |
| Pulse rate per channel | 1000 pulses/s |
| Phase duration / interphase gap | 50 / 0 microseconds |

Alpha is inherited from `src/compression.py` and must match during inversion.
Its reference uses 1000 for FS4. Adopting it here is a documented modeling choice,
not a universal CIS setting. These pulse timings are for the four-channel base
project. The 8/16/32-channel game will need a separate timing configuration.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import spectrogram
import soundfile as sf
from IPython.display import Audio, Markdown, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "cis.py").is_file():
    raise FileNotFoundError(
        "Open this notebook in the repository root on part3-reconstruction "
        "or another branch containing src/cis.py."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.audio import load_example_audio
from src.preprocessing import preprocess_audio
from src.filterbank import apply_filterbank
from src.envelope import extract_envelopes
from src.compression import compress_envelopes
from src.cis import encode_cis, plot_cis
from src.reconstruction import (
    extract_pulse_amplitudes,
    inverse_logarithmic_compression,
    interpolate_envelopes,
    noise_vocoder,
    reconstruct_audio,
)

AUDIO_FS = 22050
BAND_EDGES = [200, 500, 1250, 3150, 8000]
FILTER_ORDER = 4
ENVELOPE_CUTOFF = 400
ALPHA = 1000.0
DURATION = 8.0
STIMULATION_FS = 100000
PULSE_RATE = 1000
PHASE_DURATION_US = 50
INTERPHASE_GAP_US = 0
NOISE_SEED = 42
OUTPUT_DIR = PROJECT_ROOT / "results" / "part3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

## 2. Bridge to parts 1 and 2

This cell calls the existing functions without copying their implementations.
It uses the same eight-second speech and music segments as the original notebook.
Preprocessing removes the DC component and normalizes each input once.

The four envelopes are compressed and passed to `encode_cis`. The `CISResult`
object stores the pulse arrays and the timing metadata needed for decoding.
No `.npz` export or execution of another notebook is required.

In [ ]:
examples = [("speech", "libri1", 0.0), ("music", "brahms", 15.0)]
records = {}

for name, example, offset in examples:
    original, fs = load_example_audio(
        example, sr=AUDIO_FS, duration=DURATION, offset=offset
    )
    processed = preprocess_audio(original)
    bands = apply_filterbank(processed, fs, BAND_EDGES, order=FILTER_ORDER)
    envelopes = np.asarray(extract_envelopes(
        bands, fs, cutoff=ENVELOPE_CUTOFF, order=FILTER_ORDER
    ))
    compressed = np.asarray(compress_envelopes(envelopes, alpha=ALPHA))
    cis = encode_cis(
        compressed, fs,
        stimulation_fs=STIMULATION_FS,
        pulse_rate=PULSE_RATE,
        phase_duration_us=PHASE_DURATION_US,
        interphase_gap_us=INTERPHASE_GAP_US,
    )
    records[name] = {
        "original": processed, "fs": fs,
        "reference_envelopes": envelopes,
        "reference_compressed": compressed, "cis": cis,
    }
    print(f"{name}: {len(processed) / fs:.2f} s, "
          f"{cis.pulses.shape[0]} channels, "
          f"{cis.onset_samples.shape[1]} pulses per channel")

In [ ]:
for name, record in records.items():
    fig = plot_cis(
        record["cis"], start=0.5, duration=0.0033,
        title=f"{name.capitalize()} - CIS pulses"
    )
    fig.savefig(OUTPUT_DIR / f"{name}_cis.png", dpi=160)
    plt.show()
    plt.close(fig)

## 3. Recover samples from the pulse trains

Each pulse contains a negative phase and a positive phase of equal magnitude.
The encoder holds one compressed-envelope value across both phases. We average
only the positive phase to recover it. Averaging the whole signed pulse would
give zero because the phases cancel.

The decoder uses `onset_samples`, `phase_samples` and `gap_samples` to locate
the positive phase, including pulses with zero amplitude. It does not read
the cached `cis.amplitudes` array.

The check below independently computes the expected compressed-envelope sample
at each channel's actual pulse onset. Agreement verifies pulse decoding, not
perfect reconstruction of the original sound.

In [ ]:
for name, record in records.items():
    cis = record["cis"]
    recovered = extract_pulse_amplitudes(cis)
    audio_time = np.arange(cis.input_samples) / record["fs"]
    expected = np.array([
        np.interp(starts / cis.stimulation_fs, audio_time,
                  record["reference_compressed"][channel])
        for channel, starts in enumerate(cis.onset_samples)
    ])
    np.testing.assert_allclose(recovered, expected, rtol=0, atol=1e-12)
    record["sampled_compressed"] = recovered
    record["pulse_max_error"] = float(np.max(np.abs(recovered - expected)))
    print(f"{name}: maximum pulse-amplitude error = "
          f"{record['pulse_max_error']:.3e}")

## 4. Inverse logarithmic compression and interpolation

### Why compress and then decompress?

**Compression prepares the envelope for pulse encoding; inverse compression
returns it to the scale chosen for acoustic synthesis.** Part 1 reduces the
relative differences between weak and strong envelope values before part 2
maps them to CIS pulse amplitudes. These normalized amplitudes are a simplified
model, not calibrated patient-specific electrical currents.

After reading the actual pulses, our decoder reverses that mapping before
using the recovered envelopes to control the noise carriers. For example,
with alpha = 1000, an envelope value of 0.01 becomes approximately 0.3471 after
compression and returns to approximately 0.01 after inversion. Without the
inverse, the synthesized sound would retain the compression's change to
relative envelope levels.

**This inverse is a modeling choice for our acoustic decoder, not an operation
performed by the implant or brain in this model.** An implant delivers electrical
stimulation; our decoder creates a waveform that can be played through speakers
or headphones. Inverse compression does not recover discarded fine structure,
undo clipping, or predict an implanted listener's perception.

The encoder uses

\[
c = \frac{\ln(1+\alpha e)}{\ln(1+\alpha)}.
\]

We apply its inverse to each recovered pulse sample,

\[
\hat e = \frac{\exp(c\ln(1+\alpha))-1}{\alpha},
\]

then linearly interpolate these samples to the original audio sampling grid.
Each channel has its own staggered pulse times. For example, the default second
channel starts at 0.25 ms, not at 0 ms.

At the boundaries, the nearest recovered sample is held constant, including any
trailing partial frame omitted by the encoder. If no complete frame was encoded,
the decoder returns silence. These are explicit boundary assumptions.

The inverse is exact for the log mapping, but the overall envelope recovery is
approximate. The encoder interpolates *compressed* values before sampling, the
inverse is nonlinear, and the decoder uses linear interpolation. A 400 Hz
low-pass cutoff also does not imply a perfectly band-limited envelope.

In [ ]:
for record in records.values():
    cis = record["cis"]
    sampled = inverse_logarithmic_compression(
        record["sampled_compressed"], alpha=ALPHA
    )
    recovered_envelopes = interpolate_envelopes(
        sampled, cis.onset_samples, cis.stimulation_fs,
        record["fs"], cis.input_samples
    )
    record["sampled_envelopes"] = sampled
    record["recovered_envelopes"] = recovered_envelopes

In [ ]:
for name, record in records.items():
    time = np.arange(len(record["original"])) / record["fs"]
    fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
    errors = record["recovered_envelopes"] - record["reference_envelopes"]
    rmse = np.sqrt(np.mean(errors ** 2, axis=1))
    record["envelope_rmse"] = rmse.tolist()
    for channel, ax in enumerate(axes):
        ax.plot(time, record["reference_envelopes"][channel],
                label="Part 1 reference", color="#5266a3", linewidth=1.5)
        ax.plot(time, record["recovered_envelopes"][channel],
                label="Recovered from pulses", color="#c87543", linestyle="--")
        ax.set_ylabel(f"CH{channel + 1}")
        ax.set_xlim(0.5, 0.65)
        ax.grid(alpha=0.25)
        ax.set_title(f"{BAND_EDGES[channel]}-{BAND_EDGES[channel + 1]} Hz "
                     f"| full-record envelope RMSE = {rmse[channel]:.4g}", fontsize=10)
    axes[0].legend(loc="upper right")
    axes[-1].set_xlabel("Time [s]")
    fig.suptitle(f"{name.capitalize()} - envelope recovery (zoom)")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"{name}_envelopes.png", dpi=160)
    plt.show()
    plt.close(fig)

## 5. Noise-vocoder synthesis

For each band we generate an independent noise carrier, filter it to the same
band used in part 1 and set its RMS to one **before** modulation. We multiply
the carrier by the recovered envelope, filter the product to the same band and
sum the four synthesized channels. Normalizing each modulated output separately
would alter the relative envelope levels, so we do not do that.

The fixed seed makes the random carriers reproducible. The band-pass function
uses forward/backward filtering, so this is an offline reconstruction, not a
real-time implant processor.

Noise-envelope synthesis is based on the general approach of
[Shannon et al. (1995)](https://doi.org/10.1126/science.270.5234.303).
Our specific bands, filters and pulse-decoding steps are project choices.

In [ ]:
for name, record in records.items():
    reconstructed, channels = noise_vocoder(
        record["recovered_envelopes"], record["fs"], BAND_EDGES,
        order=FILTER_ORDER, seed=NOISE_SEED
    )
    assert reconstructed.shape == record["original"].shape
    assert np.all(np.isfinite(reconstructed))
    np.testing.assert_allclose(reconstructed, channels.sum(axis=0))
    record["reconstructed"] = reconstructed
    print(f"{name}: {len(reconstructed)} reconstructed samples, "
          f"raw peak = {np.max(np.abs(reconstructed)):.3f}")

# In the game or another script, all decoding steps can be called together:
# decoded = reconstruct_audio(cis, BAND_EDGES, alpha=ALPHA, seed=NOISE_SEED)
# raw_audio = decoded["audio"]

## 6. Original versus reconstructed audio

For playback only, the reference and reconstruction are each scaled to the same
target RMS, then attenuated by a common factor if needed to keep both peaks at
or below 0.95. Each signal receives one global gain. This does not change the
decoded envelopes and is separate from input preprocessing.

RMS matching controls signal energy, not perceived loudness. The original
reference is the preprocessed, full-band input. Differences therefore include
both vocoding and the analysis range of 200-8000 Hz.

The players use `normalize=False` so their own automatic peak normalization does
not override these gains. Start with a comfortable playback volume.

In [ ]:
def prepare_listening_pair(original, reconstructed, target_rms=0.08):
    """Apply global gains for an RMS-matched, peak-limited listening pair."""
    original_rms = np.sqrt(np.mean(original ** 2))
    reconstructed_rms = np.sqrt(np.mean(reconstructed ** 2))
    original_gain = target_rms / original_rms if original_rms > 0 else 1.0
    reconstructed_gain = target_rms / reconstructed_rms if reconstructed_rms > 0 else 1.0
    pair_peak = max(np.max(np.abs(original * original_gain)),
                    np.max(np.abs(reconstructed * reconstructed_gain)))
    safety_gain = min(1.0, 0.95 / pair_peak) if pair_peak > 0 else 1.0
    original_gain *= safety_gain
    reconstructed_gain *= safety_gain
    return (original * original_gain, reconstructed * reconstructed_gain,
            {"original": float(original_gain), "reconstructed": float(reconstructed_gain)})


for name, record in records.items():
    reference, reconstruction, gains = prepare_listening_pair(
        record["original"], record["reconstructed"]
    )
    record["listening_reference"] = reference
    record["listening_reconstruction"] = reconstruction
    record["playback_gains"] = gains
    display(Markdown(f"### {name.capitalize()}"))
    display(Markdown("**Original reference**"))
    display(Audio(reference, rate=record["fs"], normalize=False))
    display(Markdown("**Four-channel reconstruction**"))
    display(Audio(reconstruction, rate=record["fs"], normalize=False))

## 7. Waveforms and spectrograms

Both plots use the listening versions. Spectrograms share one power reference
and one color range within each pair. They show broad spectral and temporal
changes, not a speech-intelligibility score. Waveform error would not be an
appropriate fidelity target because the vocoder intentionally uses new carriers.

In [ ]:
for name, record in records.items():
    reference = record["listening_reference"]
    reconstructed = record["listening_reconstruction"]
    fs = record["fs"]
    time = np.arange(len(reference)) / fs
    fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex="col")
    signals = [reference, reconstructed]
    labels = ["Original reference", "Four-channel reconstruction"]
    spectra = [spectrogram(signal, fs=fs, window="hann", nperseg=1024,
                           noverlap=768, detrend=False, scaling="density",
                           mode="psd") for signal in signals]
    power_reference = max(float(np.max(s[2])) for s in spectra)
    power_reference = max(power_reference, np.finfo(float).tiny)
    for row, (signal, label, spectrum) in enumerate(zip(signals, labels, spectra)):
        axes[row, 0].plot(time, signal, color=["#5266a3", "#b06d3c"][row], linewidth=0.6)
        axes[row, 0].set_title(label)
        axes[row, 0].set_ylabel("Amplitude")
        axes[row, 0].set_ylim(-1, 1)
        axes[row, 0].grid(alpha=0.25)
        frequency, frame_time, power = spectrum
        power_db = 10 * np.log10(np.maximum(power / power_reference, 1e-8))
        mesh = axes[row, 1].pcolormesh(
            frame_time, frequency, power_db, shading="auto",
            cmap="magma", vmin=-80, vmax=0
        )
        axes[row, 1].set_ylim(0, fs / 2)
        axes[row, 1].set_ylabel("Frequency [Hz]")
        axes[row, 1].set_title(label)
    for ax in axes[-1]:
        ax.set_xlabel("Time [s]")
    fig.suptitle(f"{name.capitalize()} - acoustic comparison")
    fig.tight_layout(rect=[0, 0, 0.89, 0.95])
    color_axis = fig.add_axes([0.91, 0.13, 0.015, 0.68])
    fig.colorbar(mesh, cax=color_axis, label="PSD [dB relative to pair maximum]")
    fig.savefig(OUTPUT_DIR / f"{name}_comparison.png", dpi=160)
    plt.show()
    plt.close(fig)

## 8. Export WAV files and reproducibility information

The exported WAV files are the same globally scaled signals used by the players.
Raw reconstructed arrays remain available in `records`. The JSON file records
the settings, playback gains and numerical envelope checks. Rerunning this cell
replaces the corresponding files inside `results/part3/`.

In [ ]:
summary = {
    "base_commit": "19d23ee30922cd1a76abaeb40fc51b1827d00794",
    "audio_fs": AUDIO_FS, "band_edges_hz": BAND_EDGES,
    "filter_order": FILTER_ORDER, "envelope_cutoff_hz": ENVELOPE_CUTOFF,
    "alpha": ALPHA, "stimulation_fs": STIMULATION_FS,
    "pulse_rate_per_channel": PULSE_RATE,
    "phase_duration_us": PHASE_DURATION_US,
    "interphase_gap_us": INTERPHASE_GAP_US,
    "noise_seed": NOISE_SEED,
    "envelope_interpolation": "linear, nearest-value boundary hold",
    "wave_format": "PCM_16, RMS-matched pair with common peak attenuation",
    "examples": {name: {"librosa_example": example, "offset_s": offset}
                 for name, example, offset in examples},
    "checks": {},
}

for name, record in records.items():
    for label, signal in [("original", record["listening_reference"]),
                          ("reconstructed", record["listening_reconstruction"])]:
        assert np.all(np.isfinite(signal))
        assert np.max(np.abs(signal)) <= 0.9500001
        path = OUTPUT_DIR / f"{name}_{label}.wav"
        sf.write(path, signal, record["fs"], subtype="PCM_16")
        saved = sf.info(path)
        assert saved.frames == len(signal) and saved.samplerate == record["fs"]
        print(path.relative_to(PROJECT_ROOT))
    summary["checks"][name] = {
        "pulse_max_error": record["pulse_max_error"],
        "envelope_rmse_per_channel": record["envelope_rmse"],
        "playback_gains": record["playback_gains"],
        "input_samples": len(record["original"]),
    }

(OUTPUT_DIR / "reconstruction_summary.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)
print("Export complete.")

## 9. Interpretation and report notes

The numerical checks establish that the decoder reads the compressed samples
carried by the pulses and produces finite audio of the correct length. Envelope
RMSE measures agreement with the analysis envelopes. It does not measure how
well a listener understands a sentence.

When describing the method, state that inverse compression was chosen to
restore the pre-compression envelope scale for acoustic synthesis. It does
not represent a biological decoding step or a reconstruction of the original
fine structure.

For the report, add your observations after listening to both examples:

- Which words remain understandable? Is the rhythm of the speech preserved?
- For music, compare rhythm with pitch, melody and instrument identity.
- Relate these observations to the four broad bands and the replacement of
  temporal fine structure with noise.

The experiment does not include a neural model, electrode interactions,
patient-specific current thresholds or a clinical loudness map. More channels
are not guaranteed to reproduce an individual's hearing experience. These
limitations should remain explicit when connecting this work to the bonus game.

### References

- BME 576, *3.1 & 3.2*, CIS processing and auditory assignment.
- Shannon, R. V., Zeng, F.-G., Kamath, V., Wygonski, J., & Ekelid, M. (1995).
  *Speech Recognition with Primarily Temporal Cues*. Science, 270, 303-304.
  https://doi.org/10.1126/science.270.5234.303
- The alpha choice is inherited from the existing `src/compression.py`, citing
  López-Poveda et al. (2025), *Binaural audio frontend processing for cochlear
  implants inspired by the medial olivocochlear reflex*, Section 2.1.1, Eq. 1.
  https://doi.org/10.3389/fnins.2025.1678288